# Phase 2 - honest baselines per dataset (laptop version)

Reads the Phase 1 output in `D:/wifi/shared_format` (**run `phase1_shared_format_local.ipynb` first**).
The tiny CNN uses the laptop **NVIDIA GPU** if PyTorch can see it, otherwise it runs on the CPU (slower, same results).

Every model is tested only on **groups it never saw in training**:

| Dataset | Split unit (left out one at a time) | Notes |
|---|---|---|
| mat80 | setup PC-1a ... PC-4a | counts 0, 2-10 (no 1-person class) |
| countfi | scenario (head + end together) | count = one time block per scenario; expect weak results (stress test) |
| wicount | recording (15 files) | counts 0 and 3 have one file each, so a classifier can never be right on them |

**Models**
- **LightGBM** on the 10 count cues: classifier and regressor, with and without **per-group calibration**.
- **Tiny 1D-CNN** (~35k parameters, fits the ESP32-S3 budget) on the raw 5 s x 52-tone windows.
- **Mean baseline**: always predict the training-set mean.

**Levels:** `single_view` = one antenna/sub-band; `view_ensemble` = average over the views of the same moment;
`recording` = median over a whole recording.

**Needs:** `numpy pandas pyarrow matplotlib lightgbm tabulate torch`.
For the GPU, install the CUDA build of PyTorch (pytorch.org -> Get Started -> Windows / Pip / CUDA).
Cell 1 prints whether the GPU is used.

Results go to `D:/wifi/results/phase2_baselines__mat80_countfi_wicount/`.

In [ ]:
# 1. Paths, device, load Phase 1 output (laptop)
import glob
import json
import os
import time

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as Fnn

WIFI_ROOT = "D:/wifi"
SHARED = f"{WIFI_ROOT}/shared_format"           # Phase 1 output
RESULTS = f"{WIFI_ROOT}/results/phase2_baselines__mat80_countfi_wicount"
os.makedirs(RESULTS, exist_ok=True)

# ---- Phase 1 must have run first ----
for needed in ["format.json", "features.parquet"]:
    if not os.path.exists(os.path.join(SHARED, needed)):
        raise FileNotFoundError(
            f"{needed} not found in {SHARED}. Run phase1_shared_format_local.ipynb first."
        )

FORMAT = json.load(open(os.path.join(SHARED, "format.json")))
FEATURES = FORMAT["feature_names"]
FS, WIN = FORMAT["fs_hz"], int(FORMAT["win_s"] * FORMAT["fs_hz"])

feat = pd.read_parquet(os.path.join(SHARED, "features.parquet"))
feat[FEATURES] = feat[FEATURES].fillna(0.0)

DATASETS = ["mat80", "countfi", "wicount"]
MAX_COUNT = int(feat["count"].max())
SEED = 0

# ---- GPU if available, else CPU ----
if torch.cuda.is_available():
    DEVICE = "cuda"
    torch.backends.cudnn.benchmark = True
    print("device: GPU -", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    torch.set_num_threads(os.cpu_count() or 4)
    print("device: CPU (no CUDA GPU visible to PyTorch - install the CUDA build of torch to use the GPU)")

print(feat.groupby("dataset").agg(recordings=("recording", "nunique"), groups=("group", "nunique"),
                                  views=("view", "nunique"), rows=("window", "size")))

In [ ]:
# 2. Metrics and result collection
RESULT_ROWS, PER_COUNT_ROWS, RECORDING_ROWS = [], [], []


def score(y, pred):
    """pred may be continuous (regression) or integer (classification)."""
    y = np.asarray(y, dtype=float)
    pred = np.asarray(pred, dtype=float)
    rounded = np.clip(np.rint(pred), 0, MAX_COUNT)
    return {"n": len(y),
            "mae": float(np.mean(np.abs(pred - y))),
            "exact": float(np.mean(rounded == y)),
            "within1": float(np.mean(np.abs(rounded - y) <= 1))}


def record(dataset, model, task, calib, level, y, pred):
    RESULT_ROWS.append({"dataset": dataset, "model": model, "task": task, "calib": calib,
                        "level": level, **score(y, pred)})
    if level != "view_ensemble":
        return
    rounded = np.clip(np.rint(pred), 0, MAX_COUNT)
    for c in np.unique(y):
        m = y == c
        PER_COUNT_ROWS.append({"dataset": dataset, "model": model, "task": task, "calib": calib,
                               "count": int(c), "n": int(m.sum()),
                               "recall": float(np.mean(rounded[m] == c)),
                               "mean_pred": float(np.mean(pred[m]))})


def evaluate(dataset, model, task, calib, test_df, pred):
    """test_df: one row per (recording, view, window); pred aligned with it."""
    d = test_df[["recording", "window", "count"]].copy()
    d["pred"] = pred
    record(dataset, model, task, calib, "single_view", d["count"].values, d["pred"].values)

    ens = d.groupby(["recording", "window"]).agg(count=("count", "first"), pred=("pred", "mean"))
    record(dataset, model, task, calib, "view_ensemble", ens["count"].values, ens["pred"].values)

    rec = ens.groupby("recording").agg(count=("count", "first"), pred=("pred", "median"))
    record(dataset, model, task, calib, "recording", rec["count"].values, rec["pred"].values)
    for name, r in rec.iterrows():
        RECORDING_ROWS.append({"dataset": dataset, "model": model, "task": task, "calib": calib,
                               "recording": name, "count": int(r["count"]),
                               "pred": round(float(r["pred"]), 2)})

In [ ]:
# 3. LightGBM on the count cues (leave one group out) + mean baseline
GBM_PARAMS = dict(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=40,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                  random_state=SEED, verbose=-1)


def calibrate(df):
    """z-score every feature within each (group, view) using that group's own unlabelled windows."""
    out = df.copy()
    g = out.groupby(["group", "view"])[FEATURES]
    out[FEATURES] = (out[FEATURES] - g.transform("mean")) / (g.transform("std") + 1e-6)
    return out


IMPORTANCE = []
t0 = time.time()
for dataset in DATASETS:
    data = feat[feat["dataset"] == dataset].reset_index(drop=True)
    groups = sorted(data["group"].unique())

    # WiCount: one group = one recording = one count, so calibration would erase the label
    calibs = ["none"] if dataset == "wicount" else ["none", "group_z"]

    for calib in calibs:
        d = calibrate(data) if calib == "group_z" else data
        preds = {"clf": np.zeros(len(d)), "reg": np.zeros(len(d)), "mean": np.zeros(len(d))}

        for g in groups:
            test = (d["group"] == g).values
            X_tr, y_tr = d.loc[~test, FEATURES], d.loc[~test, "count"]
            X_te = d.loc[test, FEATURES]

            clf = lgb.LGBMClassifier(**GBM_PARAMS).fit(X_tr, y_tr)
            preds["clf"][test] = clf.predict(X_te)

            reg = lgb.LGBMRegressor(**GBM_PARAMS).fit(X_tr, y_tr)
            preds["reg"][test] = reg.predict(X_te)

            preds["mean"][test] = y_tr.mean()
            IMPORTANCE.append(pd.Series(reg.booster_.feature_importance("gain"), index=FEATURES,
                                        name=f"{dataset}|{calib}|{g}"))

        evaluate(dataset, "gbm", "clf", calib, d, preds["clf"])
        evaluate(dataset, "gbm", "reg", calib, d, preds["reg"])
        if calib == "none":
            evaluate(dataset, "mean_baseline", "reg", "none", d, preds["mean"])
        print(f"{dataset:8s} calib={calib:8s} done  [{time.time() - t0:.0f} s]")

gbm_table = pd.DataFrame(RESULT_ROWS)
gbm_table[gbm_table["level"] == "view_ensemble"].round(3)

In [ ]:
# 4. Tiny 1D-CNN on the raw windows (leave one group out)
EPOCHS = 12
BATCH = 256
SAMPLES_PER_EPOCH = 20000       # random subset of training windows per epoch (keeps CPU runs short)
LR = 2e-3


class TinyCNN(nn.Module):
    """Input (batch, 52 tones, 250 time bins). About 35k parameters."""

    def __init__(self, n_tones=52, n_classes=MAX_COUNT + 1):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv1d(n_tones, 32, kernel_size=7, padding=3), nn.BatchNorm1d(32), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 48, kernel_size=5, padding=4, dilation=2), nn.BatchNorm1d(48), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(48, 64, kernel_size=5, padding=4, dilation=2), nn.BatchNorm1d(64), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Dropout(0.2),
        )
        self.cls_head = nn.Linear(64, n_classes)
        self.reg_head = nn.Linear(64, 1)

    def forward(self, x):
        h = self.body(x)
        return self.cls_head(h), self.reg_head(h).squeeze(1)


def load_dataset_series(dataset):
    """All recordings of one dataset in RAM, plus an index of (recording, view, start, ...)."""
    series, index = {}, []
    for path in sorted(glob.glob(os.path.join(SHARED, dataset, "*.npz"))):
        z = np.load(path)
        meta = json.loads(str(z["meta"]))
        name = meta["recording"]
        series[name] = z["series"]                               # (views, bins, 52) float16
        for v in range(z["series"].shape[0]):
            for j, a in enumerate(z["starts"]):
                index.append((name, v, int(a), j, meta["group"], meta["count"]))
    index = pd.DataFrame(index, columns=["recording", "view", "start", "window", "group", "count"])
    return series, index


def make_batch(series, rows):
    x = np.stack([series[r.recording][r.view, r.start:r.start + WIN] for r in rows.itertuples()])
    x = np.nan_to_num(x.astype(np.float32)).clip(-3, 3)
    return torch.from_numpy(x).transpose(1, 2).to(DEVICE)       # (batch, 52, 250)


def train_and_predict(series, train_idx, test_idx):
    torch.manual_seed(SEED)
    rng = np.random.default_rng(SEED)
    model = TinyCNN().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    # classes never seen in training are masked out at prediction time
    seen = torch.zeros(MAX_COUNT + 1, dtype=torch.bool, device=DEVICE)
    seen[torch.tensor(sorted(train_idx["count"].unique()), device=DEVICE)] = True

    for epoch in range(EPOCHS):
        model.train()
        n = min(SAMPLES_PER_EPOCH, len(train_idx))
        epoch_rows = train_idx.iloc[rng.permutation(len(train_idx))[:n]]
        for b in range(0, n, BATCH):
            rows = epoch_rows.iloc[b:b + BATCH]
            y = torch.tensor(rows["count"].values, device=DEVICE)
            logits, reg = model(make_batch(series, rows))
            loss = Fnn.cross_entropy(logits, y) + Fnn.huber_loss(reg, y.float())
            opt.zero_grad()
            loss.backward()
            opt.step()
        sched.step()

    model.eval()
    p_cls, p_reg = [], []
    with torch.no_grad():
        for b in range(0, len(test_idx), 1024):
            logits, reg = model(make_batch(series, test_idx.iloc[b:b + 1024]))
            logits = logits.masked_fill(~seen, -1e9)
            p_cls.append(logits.argmax(1).cpu().numpy())
            p_reg.append(reg.cpu().numpy())
    return np.concatenate(p_cls), np.concatenate(p_reg)


t0 = time.time()
for dataset in DATASETS:
    series, index = load_dataset_series(dataset)
    pred_cls = np.zeros(len(index))
    pred_reg = np.zeros(len(index))
    for g in sorted(index["group"].unique()):
        test = (index["group"] == g).values
        pred_cls[test], pred_reg[test] = train_and_predict(series, index[~test], index[test])
        print(f"{dataset:8s} left out {g:28s} [{time.time() - t0:.0f} s]")
    evaluate(dataset, "cnn", "clf", "none", index, pred_cls)
    evaluate(dataset, "cnn", "reg", "none", index, pred_reg)
    del series

pd.DataFrame(RESULT_ROWS).query("model == 'cnn' and level == 'view_ensemble'").round(3)

In [ ]:
# 5. Save all results + summary tables
results = pd.DataFrame(RESULT_ROWS)
per_count = pd.DataFrame(PER_COUNT_ROWS)
recordings = pd.DataFrame(RECORDING_ROWS)
importance = pd.concat(IMPORTANCE, axis=1)

results.to_csv(os.path.join(RESULTS, "results.csv"), index=False)
per_count.to_csv(os.path.join(RESULTS, "per_count_recall.csv"), index=False)
recordings.to_csv(os.path.join(RESULTS, "recording_predictions.csv"), index=False)
importance.to_csv(os.path.join(RESULTS, "gbm_feature_importance.csv"))

summary = (results.pivot_table(index=["dataset", "model", "task", "calib"], columns="level",
                               values=["mae", "within1", "exact"])
           .round(3))
summary.to_csv(os.path.join(RESULTS, "summary.csv"))
summary

In [ ]:
# 6. Plots: per-count recall and recording-level predictions
fig_dir = os.path.join(RESULTS, "figures")
os.makedirs(fig_dir, exist_ok=True)

# per-count recall (view ensemble) for every model of every dataset
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, dataset in zip(axes, DATASETS):
    d = per_count[per_count["dataset"] == dataset]
    for (model, task, calib), h in d.groupby(["model", "task", "calib"]):
        ax.plot(h["count"], h["recall"], marker="o", label=f"{model}/{task}/{calib}")
    ax.set_title(f"{dataset}: per-count recall (view ensemble)")
    ax.set_xlabel("people")
    ax.set_ylim(0, 1)
    ax.legend(fontsize=6)
fig.tight_layout()
fig.savefig(os.path.join(fig_dir, "per_count_recall.png"), dpi=120)
plt.show()

# recording-level: true vs predicted count (regression models)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, dataset in zip(axes, DATASETS):
    d = recordings[(recordings["dataset"] == dataset) & (recordings["task"] == "reg")]
    for (model, calib), h in d.groupby(["model", "calib"]):
        ax.scatter(h["count"], h["pred"], s=18, label=f"{model}/{calib}", alpha=0.8)
    lim = [0, MAX_COUNT]
    ax.plot(lim, lim, "k--", lw=0.8)
    ax.set_title(f"{dataset}: whole-recording estimate")
    ax.set_xlabel("true people")
    ax.set_ylabel("predicted (median over recording)")
    ax.legend(fontsize=6)
fig.tight_layout()
fig.savefig(os.path.join(fig_dir, "recording_predictions.png"), dpi=120)
plt.show()

# mean feature importance per dataset
imp = importance.copy()
imp.columns = [c.split("|")[0] for c in imp.columns]
imp = imp.T.groupby(level=0).mean().T
(imp / imp.sum()).round(3)

In [ ]:
# 7. Short markdown report on Drive (results/phase2_baselines__mat80_countfi_wicount/PHASE2_RESULTS.md)
ens = results[results["level"] == "view_ensemble"].copy()
best = ens.sort_values("mae").groupby("dataset").head(1)

lines = ["# Phase 2 results - honest held-out baselines", "",
         f"Generated {time.strftime('%Y-%m-%d %H:%M')} from shared_format ({FORMAT['win_s']} s windows, "
         f"{FORMAT['fs_hz']} Hz, {FORMAT['n_tones']} tones).", "",
         "## Best model per dataset (view ensemble)", "",
         best[["dataset", "model", "task", "calib", "mae", "exact", "within1", "n"]]
         .round(3).to_markdown(index=False), "",
         "## All models (view ensemble)", "",
         ens[["dataset", "model", "task", "calib", "mae", "exact", "within1"]]
         .round(3).to_markdown(index=False), "",
         "## Single view (what one ESP32 link sees)", "",
         results[results["level"] == "single_view"][["dataset", "model", "task", "calib", "mae", "within1"]]
         .round(3).to_markdown(index=False), "",
         "## Caveats", "",
         "- mat80 has no 1-person recordings; PC4a_n00 looked occupied in the deep audit.",
         "- countfi: each count is one time block per scenario, so results measure scenario transfer only.",
         "- wicount: counts 0 and 3 have one file each; a left-out file of those counts was never seen in training.",
         "- group_z calibration uses all unlabelled windows of the test group (slightly optimistic)."]

with open(os.path.join(RESULTS, "PHASE2_RESULTS.md"), "w") as f:
    f.write(chr(10).join(lines))
print("saved", os.path.join(RESULTS, "PHASE2_RESULTS.md"))
best

In [ ]:
# 8. Done
print("Phase 2 finished. Results in", RESULTS)